In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torchaudio
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

# --- Paths Configuration ---
BASE_PATH = "/kaggle/input/competitions/digitrecognition-ee708" 
TRAIN_AUDIO_DIR = os.path.join(BASE_PATH, "train_audio", "train_audio")
TEST_AUDIO_DIR = os.path.join(BASE_PATH, "test_audio", "test_audio")
TRAIN_CSV = os.path.join(BASE_PATH, "train.csv")

# --- Hyperparameters ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 32
EPOCHS = 15 
LR = 0.001
SAMPLE_RATE = 16000
MAX_LEN = 16000
N_MELS = 64

print(f"Using device: {DEVICE}")

Using device: cuda


In [2]:
class AdvancedDigitDataset(Dataset):
    def __init__(self, df, audio_dir, mode="train"):
        self.df = df
        self.audio_dir = audio_dir
        self.mode = mode
        
        self.mel_spectrogram = torchaudio.transforms.MelSpectrogram(
            sample_rate=SAMPLE_RATE, n_mels=N_MELS
        )
        self.amplitude_to_db = torchaudio.transforms.AmplitudeToDB()
        
        # SpecAugment: Only apply during training
        if self.mode == "train":
            self.freq_masking = torchaudio.transforms.FrequencyMasking(freq_mask_param=15)
            self.time_masking = torchaudio.transforms.TimeMasking(time_mask_param=35)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        audio_id = str(self.df.iloc[idx]['id']).strip()
        audio_filename = audio_id if audio_id.endswith('.wav') else f"{audio_id}.wav"
        audio_path = os.path.join(self.audio_dir, audio_filename)
        
        waveform, sr = torchaudio.load(audio_path)
        
        if sr != SAMPLE_RATE:
            waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)(waveform)
            
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
            
        if waveform.shape[1] > MAX_LEN:
            waveform = waveform[:, :MAX_LEN]
        elif waveform.shape[1] < MAX_LEN:
            waveform = torch.nn.functional.pad(waveform, (0, MAX_LEN - waveform.shape[1]))
            
        mel_spec = self.mel_spectrogram(waveform)
        mel_spec = self.amplitude_to_db(mel_spec) 
        
        # Apply masks only to the training data
        if self.mode == "train":
            mel_spec = self.freq_masking(mel_spec)
            mel_spec = self.time_masking(mel_spec)
        
        if self.mode == "test":
            return mel_spec, audio_id
        else:
            return mel_spec, int(self.df.iloc[idx]['label'])

In [3]:
class RobustAudioCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(RobustAudioCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            
            # Squashes output to 1x1 dynamically
            nn.AdaptiveAvgPool2d((1, 1)) 
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(), 
            nn.Dropout(0.5), # Prevents memorizing specific speaker characteristics
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = RobustAudioCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

In [4]:
# 1. Load data and split into Train (80%) and Validation (20%)
full_train_df = pd.read_csv(TRAIN_CSV)
train_df, valid_df = train_test_split(full_train_df, test_size=0.2, random_state=42, stratify=full_train_df['label'])

# 2. Create DataLoaders
train_dataset = AdvancedDigitDataset(train_df, TRAIN_AUDIO_DIR, mode="train")
valid_dataset = AdvancedDigitDataset(valid_df, TRAIN_AUDIO_DIR, mode="valid")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False)

# 3. Training Loop
best_f1_score = 0.0

print("Starting Robust Training Loop...")
for epoch in range(EPOCHS):
    # --- TRAINING PHASE ---
    model.train()
    train_loss = 0.0
    
    for specs, labels in train_loader:
        specs, labels = specs.to(DEVICE), labels.to(DEVICE)
        
        optimizer.zero_grad()
        outputs = model(specs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # --- VALIDATION PHASE ---
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for specs, labels in valid_loader:
            specs, labels = specs.to(DEVICE), labels.to(DEVICE)
            outputs = model(specs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(valid_loader)
    val_macro_f1 = f1_score(all_targets, all_preds, average='macro')
    
    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Macro F1: {val_macro_f1:.4f}")
    
    # Save best model
    if val_macro_f1 > best_f1_score:
        best_f1_score = val_macro_f1
        torch.save(model.state_dict(), '/kaggle/working/best_model.pth')
        print(f"  -> Saved new best model with F1: {best_f1_score:.4f}")

print("\nTraining complete! Best Macro F1 score:", round(best_f1_score, 4))

Starting Robust Training Loop...
Epoch [1/15] | Train Loss: 1.8058 | Val Loss: 1.1022 | Val Macro F1: 0.6026
  -> Saved new best model with F1: 0.6026
Epoch [2/15] | Train Loss: 1.2537 | Val Loss: 0.6314 | Val Macro F1: 0.7613
  -> Saved new best model with F1: 0.7613
Epoch [3/15] | Train Loss: 0.9536 | Val Loss: 0.3517 | Val Macro F1: 0.8786
  -> Saved new best model with F1: 0.8786
Epoch [4/15] | Train Loss: 0.7522 | Val Loss: 0.2071 | Val Macro F1: 0.9326
  -> Saved new best model with F1: 0.9326
Epoch [5/15] | Train Loss: 0.6175 | Val Loss: 0.2299 | Val Macro F1: 0.9181
Epoch [6/15] | Train Loss: 0.5397 | Val Loss: 0.1251 | Val Macro F1: 0.9630
  -> Saved new best model with F1: 0.9630
Epoch [7/15] | Train Loss: 0.4762 | Val Loss: 0.1684 | Val Macro F1: 0.9414
Epoch [8/15] | Train Loss: 0.4381 | Val Loss: 0.1001 | Val Macro F1: 0.9661
  -> Saved new best model with F1: 0.9661
Epoch [9/15] | Train Loss: 0.4121 | Val Loss: 0.1143 | Val Macro F1: 0.9572
Epoch [10/15] | Train Loss: 0.3

In [5]:
print("Loading best model weights for test inference...")
model.load_state_dict(torch.load('/kaggle/working/best_model.pth'))
model.eval()

# Load test files directly from directory
test_files = [f for f in os.listdir(TEST_AUDIO_DIR) if f.endswith('.wav')]
if len(test_files) == 0:
    raise ValueError(f"CRITICAL ERROR: No .wav files found in {TEST_AUDIO_DIR}")

test_ids = [f.replace('.wav', '') for f in test_files]
test_df = pd.DataFrame({'id': test_ids})

test_dataset = AdvancedDigitDataset(test_df, TEST_AUDIO_DIR, mode="test")
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

submission_ids = []
submission_labels = []

print(f"Running inference on {len(test_df)} test files...")
with torch.no_grad():
    for specs, audio_ids in test_loader:
        specs = specs.to(DEVICE)
        outputs = model(specs)
        _, predicted = torch.max(outputs.data, 1)
        
        for i in range(len(audio_ids)):
            submission_ids.append(audio_ids[i])
            submission_labels.append(predicted[i].item())

submission_df = pd.DataFrame({
    'id': submission_ids,
    'label': submission_labels
})

submission_df.to_csv('/kaggle/working/submission.csv', index=False)
print("Saved /kaggle/working/submission.csv successfully!")
print("Ready to commit notebook and submit!")

Loading best model weights for test inference...
Running inference on 16200 test files...
Saved /kaggle/working/submission.csv successfully!
Ready to commit notebook and submit!
